##Streaming the weather data for every 30 sec

In [0]:

from azure.eventhub import EventHubProducerClient, EventData
import json
import requests
from datetime import datetime, timedelta

# Event Hub configuration
EVENT_HUB_NAME = "weatherstreamingeventhub"
eventhub_connection_string = dbutils.secrets.get(scope="key-vault-scope", key="eventhub-connection-string")
weatherapikey = dbutils.secrets.get(scope="key-vault-scope", key="weatherapikey")

# Initialize the Event Hub producer
producer = EventHubProducerClient.from_connection_string(conn_str=eventhub_connection_string, eventhub_name=EVENT_HUB_NAME)

# Function to send events to Event Hub
def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)

# Function to handle the API response
def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

# Function to get current weather and air quality data
def get_current_weather(base_url, api_key, location):
    current_weather_url = f"{base_url}/current.json"
    params = {
        'key': api_key,
        'q': location,
        "aqi": 'yes'
    }
    response = requests.get(current_weather_url, params=params)
    return handle_response(response)

# Function to get Forecast Data
def get_forecast_weather(base_url, api_key, location, days):
    forecast_url = f"{base_url}/forecast.json"
    params = {
        "key": api_key,
        "q": location,
        "days": days,
    }
    response = requests.get(forecast_url, params=params)
    return handle_response(response)

# Function to get Alerts
def get_alerts(base_url, api_key, location):
    alerts_url = f"{base_url}/forecast.json"
    params = {
        'key': api_key,
        'q': location,
        "alerts": 'yes'
    }
    response = requests.get(alerts_url, params=params)
    return handle_response(response)

# Flatten and merge the data
def flatten_data(current_weather, forecast_weather, alerts):
    location_data = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    flattened_data = {
        'name': location_data.get('name'),
        'region': location_data.get('region'),
        'country': location_data.get('country'),
        'lat': location_data.get('lat'),
        'lon': location_data.get('lon'),
        'localtime': location_data.get('localtime'),
        'temp_c': current.get('temp_c'),
        'is_day': current.get('is_day'),
        'condition_text': condition.get('text'),
        'condition_icon': condition.get('icon'),
        'wind_kph': current.get('wind_kph'),
        'wind_degree': current.get('wind_degree'),
        'wind_dir': current.get('wind_dir'),
        'pressure_in': current.get('pressure_in'),
        'precip_in': current.get('precip_in'),
        'humidity': current.get('humidity'),
        'cloud': current.get('cloud'),
        'feelslike_c': current.get('feelslike_c'),
        'uv': current.get('uv'),
        'air_quality': {
            'co': air_quality.get('co'),
            'no2': air_quality.get('no2'),
            'o3': air_quality.get('o3'),
            'so2': air_quality.get('so2'),
            'pm2_5': air_quality.get('pm2_5'),
            'pm10': air_quality.get('pm10'),
            'us-epa-index': air_quality.get('us-epa-index'),
            'gb-defra-index': air_quality.get('gb-defra-index')
        },
        'alerts': [
            {
                'headline': alert.get('headline'),
                'severity': alert.get('severity'),
                'description': alert.get('desc'),
                'instruction': alert.get('instruction')
            }
            for alert in alert_list
        ],
        'forecast': [
            {
                'date': day.get('date'),
                'maxtemp_c': day.get('day', {}).get('maxtemp_c'),
                'mintemp_c': day.get('day', {}).get('mintemp_c'),
                'condition': day.get('day', {}).get('condition', {}).get('text')
            }
            for day in forecast
        ]
    }
    return flattened_data


def fetch_weather_data():

    base_url = "http://api.weatherapi.com/v1/"
    location = "Chennai"  # You can replace with any city name based on your preference
    weatherapikey = dbutils.secrets.get(scope="key-vault-scope", key="weatherapikey")

    # Get data from API
    current_weather = get_current_weather(base_url, weatherapikey, location)
    forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
    alerts = get_alerts(base_url, weatherapikey, location)

    # Flatten and merge data
    merged_data = flatten_data(current_weather, forecast_weather, alerts)
    return merged_data

# Function to process each batch of streaming data
last_sent_time = datetime.now() - timedelta(seconds=30)  # Initialize last sent time


# Main program
def process_batch(batch_df, batch_id):
    global last_sent_time
    try:
        # Get current time
        current_time = datetime.now()
        
        # Check if 30 seconds have passed since last event was sent
        if (current_time - last_sent_time).total_seconds() >= 30:
            # Fetch weather data
            weather_data = fetch_weather_data()
            
            # Send the weather data (current weather part)
            send_event(weather_data)

            # Update last sent time
            last_sent_time = current_time
            print(f'Event Sent at {last_sent_time}')

    except Exception as e:
        print(f"Error sending events in batch {batch_id}: {str(e)}")
        raise e

# Set up a streaming source (for example, rate source for testing purposes)
streaming_df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

# Write the streaming data using foreachBatch to send weather data to Event Hub
query = streaming_df.writeStream.option("checkpointLocation", "abfss://checkpointloc@weatherstreamingcheck.dfs.core.windows.net/checkpoints").foreachBatch(process_batch).start()

query.awaitTermination()

# Close the producer after termination
producer.close()

Event Sent at 2026-09-01 11:11:45.619085
Event Sent at 2026-09-01 11:12:16.334623
Event Sent at 2026-09-01 11:12:46.522228
Event Sent at 2026-09-01 11:13:17.357287
Event Sent at 2026-09-01 11:13:47.414748
Event Sent at 2026-09-01 11:14:17.605783
Event Sent at 2026-09-01 11:14:48.440184
Event Sent at 2026-09-01 11:15:18.832835
Event Sent at 2026-09-01 11:15:49.572275
Event Sent at 2026-09-01 11:16:19.737270
Event Sent at 2026-09-01 11:16:50.553067
Event Sent at 2026-09-01 11:17:21.335933
Event Sent at 2026-09-01 11:17:52.319331
Event Sent at 2026-09-01 11:18:22.456240
Event Sent at 2026-09-01 11:18:53.365843
Event Sent at 2026-09-01 11:19:23.733478
Event Sent at 2026-09-01 11:19:54.692889
Event Sent at 2026-09-01 11:20:25.328874
Event Sent at 2026-09-01 11:20:55.511033
Event Sent at 2026-09-01 11:21:26.285201
Event Sent at 2026-09-01 11:21:56.433357
Event Sent at 2026-09-01 11:22:26.581061
Event Sent at 2026-09-01 11:22:57.337550
Event Sent at 2026-09-01 11:23:27.755391
Event Sent at 20

Event Sent at 2026-09-01 11:25:00.416423
Event Sent at 2026-09-01 11:25:30.496809
Event Sent at 2026-09-01 11:26:01.313426
Event Sent at 2026-09-01 11:26:32.301912
Event Sent at 2026-09-01 11:27:02.676420
Event Sent at 2026-09-01 11:27:32.693396
Event Sent at 2026-09-01 11:28:03.326019
Event Sent at 2026-09-01 11:28:33.724813
Event Sent at 2026-09-01 11:29:04.482798
Event Sent at 2026-09-01 11:29:35.489220
Event Sent at 2026-09-01 11:30:05.493836
Event Sent at 2026-09-01 11:30:36.443658
Event Sent at 2026-09-01 11:31:06.711991
Event Sent at 2026-09-01 11:31:37.619850
Event Sent at 2026-09-01 11:32:08.285607
Event Sent at 2026-09-01 11:32:38.517125
Event Sent at 2026-09-01 11:33:08.713471
Event Sent at 2026-09-01 11:33:39.239689
Event Sent at 2026-09-01 11:34:09.639667
Event Sent at 2026-09-01 11:34:40.266442
Event Sent at 2026-09-01 11:35:10.398986
Event Sent at 2026-09-01 11:35:40.676918
Event Sent at 2026-09-01 11:36:11.636323


Event Sent at 2026-09-01 11:36:41.651849
Event Sent at 2026-09-01 11:37:12.332264
Event Sent at 2026-09-01 11:37:42.439559
Event Sent at 2026-09-01 11:38:12.503566
Event Sent at 2026-09-01 11:38:43.289563
Event Sent at 2026-09-01 11:39:13.672242
Event Sent at 2026-09-01 11:39:43.673472
Event Sent at 2026-09-01 11:40:13.688528
Event Sent at 2026-09-01 11:40:44.562333
Event Sent at 2026-09-01 11:41:14.615036
Event Sent at 2026-09-01 11:41:45.583611
Event Sent at 2026-09-01 11:42:16.507260
Event Sent at 2026-09-01 11:42:47.376980
Event Sent at 2026-09-01 11:43:17.585839
Event Sent at 2026-09-01 11:43:47.673811


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/spark/python